In [ ]:
import numpy as np

n=60_000
mat = np.random.rand(n,n)
print(
    mat[:2, :2]
)

def symmetrize(mat: np.ndarray) -> np.ndarray:
    return mat + np.tril(mat, -1).T

def symmetrize_inplace(mat: np.ndarray, block_size: int = 5000) -> np.ndarray:
    n = mat.shape[0]
    for i in range(0, n, block_size):
        for j in range(i, n, block_size):
            if i == j:
                # Blocks on diagonal
                block = mat[i:i+block_size, i:i+block_size]
                triu_idx = np.triu_indices(block.shape[0], k=1)
                block[triu_idx] = block.T[triu_idx]
            else:
                # Off-diag blocks
                mat[i:i+block_size, j:j+block_size] = mat[j:j+block_size, i:i+block_size].T

symmetrize_inplace(mat)

print(
    mat[:2, :2]
)

[[0.43874026 0.16687433]
 [0.37140281 0.25054284]]
[[0.43874026 0.37140281]
 [0.37140281 0.25054284]]


In [2]:
del mat

In [10]:
mat = np.random.rand(3, 3)
ilow = np.tril_indices_from(mat, -1)
ihi = np.triu_indices_from(mat, 1)

In [12]:
mat

array([[0.67289682, 0.92911432, 0.62313908],
       [0.97141419, 0.76639039, 0.18047791],
       [0.34175734, 0.60698548, 0.96145529]])

In [13]:
mat[ihi] = mat[ilow]
mat

array([[0.67289682, 0.97141419, 0.34175734],
       [0.97141419, 0.76639039, 0.60698548],
       [0.34175734, 0.60698548, 0.96145529]])

In [1]:
import numpy as np
import psutil
import os
import gc
import time

process = psutil.Process(os.getpid())

def rss_gb():
    return process.memory_info().rss / 1024**3

def benchmark_symmetrize(n=20000, dtype=np.float64):
    print(f"Allocating {n} x {n} matrix ({dtype})...")
    A = np.triu(np.random.rand(n, n).astype(dtype))
    gc.collect()
    print(f"Initial RSS: {rss_gb():.3f} GB")

    # --- Old version ---
    B = A.copy()
    gc.collect()
    print("\nTesting old symmetrize...")
    print(f"Before: {rss_gb():.3f} GB")
    t0 = time.perf_counter()

    B = B + np.tril(B, -1).T

    t1 = time.perf_counter()
    print(f"After:  {rss_gb():.3f} GB")
    print(f"Elapsed: {t1 - t0:.3f} s")

    del B
    gc.collect()
    print(f"After cleanup: {rss_gb():.3f} GB")

    # --- In-place version ---
    C = A.copy()
    gc.collect()
    print("\nTesting in-place symmetrize...")
    print(f"Before: {rss_gb():.3f} GB")
    t0 = time.perf_counter()

    idx = np.tril_indices_from(C, -1)
    C.T[idx] = C[idx]

    t1 = time.perf_counter()
    print(f"After:  {rss_gb():.3f} GB")
    print(f"Elapsed: {t1 - t0:.3f} s")

    del C, A, idx
    gc.collect()
    print(f"After cleanup: {rss_gb():.3f} GB")

benchmark_symmetrize(n=20000)  # increase to 20000 if you have enough RAM

Allocating 20000 x 20000 matrix (<class 'numpy.float64'>)...
Initial RSS: 3.067 GB

Testing old symmetrize...
Before: 6.047 GB
After:  6.047 GB
Elapsed: 4.100 s
After cleanup: 3.067 GB

Testing in-place symmetrize...
Before: 6.047 GB
After:  9.027 GB
Elapsed: 3.342 s
After cleanup: 0.087 GB


In [3]:
rss_gb()
A = np.random.rand(10000, 10000)

In [14]:
idx = np.tril_indices_from(A, -1)
A.T[idx] = A[idx]

In [20]:
rss_gb()

1.5772209167480469

In [8]:
import numpy as np

In [9]:
mat = np.array([
    [1.0, 0.0, 0.0],
    [2.0, 4.0, 0.0],
    [3.0, 5.0, 6.0]
])

In [3]:
def symmetrize_in_place(mat: np.ndarray) -> np.ndarray:
        i_lower = np.tril_indices_from(mat)
        mat.T[i_lower] = mat[i_lower]

In [ ]:
mat = symmetrize_in_place(mat) # It returns nothing

In [7]:
mat